# validation check — KMeans (running) -> profitable clusters -> LR, DRAW market

`points` space (2D: home vs away season points-per-match average), draw market. One segmentation method: KMeans on the running season
features, keeping the clusters that beat the market on the training fold. The k-sweep at the
bottom is an extra check, outside the template.

In [1]:
# --- SETUP -------------------------------------------------------------------
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

from evaluation.model_check import run_check, collect_bets, pooled_roi, roi_grouped, season_folds

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [16]:
# --- SPACE (modifiable) — the model space, defined in one place --------------
# The MODEL space: the features the bet-signal model reads. This is the ONLY cell to change
# to move the model to another space. (The segmentation space is set in the SEGMENTER cell.)
FEATURES = [
    "hmt_season_impl_points_avg",
    "awt_season_impl_points_avg",
    "hmt_season_points_avg",
    "awt_season_points_avg", 'mrkt_draw_impl'
]

In [17]:
# --- TARGET (modifiable) -----------------------------------------------------
# OUTCOME is the 0/1 event we bet on. IMPLIED is the market-consensus implied probability
# (mrkt), used for both the profitability check and the ROI — one odds source, never mixed.
OUTCOME = "t_draw_flg"          # 1 when the match ended in a draw
IMPLIED = "mrkt_draw_impl"      # implied draw prob = 1 / mrkt draw odds (profitability + betting)

In [18]:
# --- ABT FILTERING -----------------------------------------------------------
# Keep matches where both teams have enough history for stable season averages; drop rows missing any
# feature or the target.
MIN_GAME_NUMBER = 8
matches = abt[
    (abt["hmt_game_number"] > MIN_GAME_NUMBER)
    & (abt["awt_game_number"] > MIN_GAME_NUMBER)
].dropna(subset=FEATURES + [IMPLIED, OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 5533   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england', 'france', 'germany', 'italy', 'spain']


In [24]:
# --- SEGMENTER (modifiable) — KMeans on the running space --------------------
# SEGMENT_FEATURES is the space the SEGMENTATION uses. It defaults to the model's FEATURES (segment on
# the same variables the model sees); override it with a different list to segment on other variables
# than the model. The model space (FEATURES, used by run_check below) is unaffected.
SEGMENT_FEATURES = FEATURES

# The method IS this notebook's identity; a different method = a different notebook. Here: fit KMeans on
# SEGMENT_FEATURES (train only), keep the chosen clusters (keep="profitable" or "best"), label each row
# by its kept cluster (NaN outside -> not bet). Contract: segmenter(train) -> labeler ; labeler -> Series.

# Per-segment edge on the TRAIN fold only, so the held-out fold never influences the choice.
#   keep="profitable" -> keep every positive-edge segment;   keep="best" -> keep only the single
# highest-edge one. Swap it via the `keep=` argument of kmeans_segmenter below.
def keep_segment(train, labels, keep="profitable", min_matches=100):
    frame = pd.DataFrame({"label": labels.to_numpy(),
                          "won": train[OUTCOME].to_numpy(),
                          "implied": train[IMPLIED].to_numpy()})
    stats = frame.groupby("label").agg(won=("won", "mean"), implied=("implied", "mean"), n=("won", "size"))
    edge = (stats["won"] - stats["implied"])[stats["n"] >= min_matches]
    if keep == "best":
        return {edge.idxmax()} if len(edge) else set()   # single highest-edge segment (sign ignored)
    return set(edge[edge > 0].index)                                    # all positive-edge segments

def kmeans_segmenter(features, n_clusters, keep="profitable", min_matches=100):
    def segmenter(train):
        km = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(train[features].to_numpy())
        labels = pd.Series(km.predict(train[features].to_numpy()), index=train.index)
        kept = keep_segment(train, labels, keep=keep, min_matches=min_matches)
        def labeler(rows):
            c = pd.Series(km.predict(rows[features].to_numpy()), index=rows.index)
            return c.where(c.isin(kept))     # non-kept clusters -> NaN -> not bet
        return labeler
    return segmenter

K = 8   # cluster count (see the k-sweep extra check below)
segmenter = kmeans_segmenter(SEGMENT_FEATURES, K)   # keep="best" to bet only the single best cluster

# The segmenter's self-description: fit on all matches (in-sample) and show each cluster's edge — size,
# outcome rate vs implied, ROI, centre (in SEGMENT_FEATURES). In-sample: a map of WHERE the edge sits,
# not gated evidence. KEEP marks clusters a training fold would keep (positive edge).
_km = KMeans(n_clusters=K, random_state=0, n_init=10).fit(matches[SEGMENT_FEATURES].to_numpy())
_lab = _km.predict(matches[SEGMENT_FEATURES].to_numpy())
print(f"segment edges (in-sample, k={K})   centre = [{'  '.join(SEGMENT_FEATURES)}]")
print(" seg      n   rate%  impl%   edge     ROI   keep   centre")
for s in range(K):
    m = matches[_lab == s]
    dr, im = m[OUTCOME].mean(), m[IMPLIED].mean()
    centre = " ".join(f"{x:.2f}" for x in _km.cluster_centers_[s])
    print(f" {s:>3} {len(m):>7}  {dr*100:5.1f}  {im*100:5.1f}  {(dr-im)*100:+5.1f}  "
          f"{pooled_roi(m[IMPLIED], m[OUTCOME].astype(int)):+6.3f}  {'KEEP' if dr > im else '    '}  [{centre}]")

segment edges (in-sample, k=8)   centre = [hmt_season_impl_points_avg  awt_season_impl_points_avg  hmt_season_points_avg  awt_season_points_avg  mrkt_draw_impl]
 seg      n   rate%  impl%   edge     ROI   keep   centre
   0     920   27.4   24.1   +3.3  +0.137  KEEP  [1.58 1.14 1.59 0.93 0.24]
   1     424   20.5   23.5   -3.0  -0.127        [1.33 2.19 1.25 2.33 0.24]
   2     554   13.7   17.9   -4.2  -0.234        [2.08 1.24 2.22 1.08 0.18]
   3    1032   27.6   29.7   -2.1  -0.071        [1.15 1.32 0.95 1.27 0.30]
   4     782   24.7   27.7   -3.0  -0.108        [1.54 1.57 1.55 1.58 0.28]
   5     727   30.5   28.6   +1.9  +0.066  KEEP  [1.14 1.07 0.93 0.77 0.29]
   6     685   26.3   26.4   -0.2  -0.006        [1.13 1.76 0.90 1.83 0.26]
   7     409   21.5   25.9   -4.3  -0.168        [1.99 1.88 2.10 1.98 0.26]


In [25]:
# --- MODEL (modifiable) ------------------------------------------------------
# The bet-signal model: any sklearn-style estimator with .fit(X, y) and .predict_proba(X). It is fit
# on the MODEL space (FEATURES) — independent of the segmentation space above.
def make_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                         StandardScaler(), LogisticRegression(max_iter=3000))

BUFFER = 0.02   # only bet when the model probability beats the implied one by at least this margin

## Summary
The three gates at a glance, and the one-line row written to `model_checks/results.md`.

In [26]:
_ = run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=segmenter, make_model=make_model, buffer=BUFFER,
              source="points_draw_kmeans_match_lr", space="points", target="draw",
              abt_filter="major, gn>8, dropna",
              segmentation=f"KMeans k={K} (match, running, {len(SEGMENT_FEATURES)}D)",
              segment_filter="train-profitable clusters")

points_draw_kmeans_match_lr   pooled +0.120 (1012 bets)   [mrkt]

GATE 1  LOSO — ROI by test season           PASS  (+4/4 seasons)
    2223 +0.153    2324 +0.136    2425 +0.103    2526 +0.090

GATE 2  league — ROI by league / drop best   PASS
    germany +0.204    england +0.163    spain +0.105    italy +0.075    france +0.067
    pooled without best league (germany): +0.102

GATE 3  walk-forward — ROI by test season    PASS
    2425 +0.074    2526 +0.090


## Extra checks — below the template
Everything above is the standard template (config cells + Summary). Cells below are notebook-specific
diagnostics, outside the template — they may use knobs a given segmenter has (e.g. a cluster count)
that other methods do not.

In [23]:
# EXTRA CHECK (below the template) — knob robustness: sweep the cluster count k = 2..10.
# Recomputes the whole honest per-fold pipeline for each k (segmenting on SEGMENT_FEATURES, model on
# FEATURES). A real edge survives a range of k; one that appears at a single k is fragile.
print(" k    pooled    profit   bets   seasons+")
for k in range(2, 15):
    b = collect_bets(matches, season_folds(matches), features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
                     segmenter=kmeans_segmenter(SEGMENT_FEATURES, k), make_model=make_model, buffer=BUFFER)
    ps = roi_grouped(b, "fold")
    print(f" {k:<3} {pooled_roi(b['implied'], b['won']):+.4f}  {(b['won'] - b['implied']).sum():+7.2f}  {len(b):>4}   +{int((ps > 0).sum())}/{len(ps)}")

 k    pooled    profit   bets   seasons+
 2   -0.1149    -7.53   252   +0/1
 3   +0.0863   +10.73   464   +1/2
 4   +0.0548    +6.81   451   +1/2
 5   +0.0869   +17.51   787   +4/4
 6   +0.1178   +26.76   876   +4/4
 7   +0.1357   +29.52   824   +4/4
 8   +0.1197   +31.43  1012   +4/4
 9   +0.0664   +13.63   793   +4/4
 10  +0.1032   +24.23   924   +4/4
 11  +0.1080   +24.17   878   +4/4
 12  +0.0626   +13.66   855   +3/4
 13  +0.0611   +14.11   894   +4/4
 14  +0.1026   +25.97   990   +4/4
